# 09d · stereoscope vs cell2location — cross-method check of the microglia signal

**Question:** does the microglia result (fraction maps, hotspots, and the downstream homeostatic/DAM state shift from notebooks 10/10c) depend on cell2location, or does an independent model reproduce it?

**Setup:** both runs use the same reference (6 Linnarsson brain classes + `MYE:Microglia`, see run 3 in 09/09b) and the same gene set. Stereoscope runs via `utils/run_deconv_stereoscope.py` on the peer (env `c2l`, `scvi.external`). Its outputs go to `results/tables/deconvolution_stereoscope/`; pull them back before running this notebook.

**Checks**
1. Per-spot agreement of cell-type fractions (focus: microglia).
2. Section-level microglia means, including the P24215 vs P28052 slide pattern.
3. Hotspot overlap (global z > 1.5, as in notebook 10).
4. Downstream: deconv-weighted section means (10c-A) and hotspot mixed models with and without slide adjustment (10/11), recomputed with each method's weights.

Everything runs locally from CSVs; signature scores come from `results/inflammation/tables/per_spot_scores.csv`.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import mannwhitneyu, pearsonr, spearmanr

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    ROOT = ROOT.parent
TBL = ROOT / 'results' / 'tables'
C2L_CSV    = TBL / 'deconvolution_microglia' / 'per_spot_fractions.csv'
STEREO_CSV = TBL / 'deconvolution_stereoscope' / 'per_spot_fractions.csv'
NOISE_CSV  = TBL / 'deconvolution_stereoscope' / 'noise_fraction.csv'
SCORES_CSV = ROOT / 'results' / 'inflammation' / 'tables' / 'per_spot_scores.csv'
OUT = TBL / 'deconvolution_stereoscope'
FIG = ROOT / 'results' / 'figures' / 'manuscript'

SAMPLE_KEY, TREAT_KEY, REG = 'sample_id', 'treatment', 're_annotation_regions'
MG = 'MYE:Microglia'
ORDER = ['WT', 'PBS', 'BRICHOS']; TCOL = {'WT': '#7570b3', 'PBS': '#d95f02', 'BRICHOS': '#1b9e77'}
Z_THRESH = 1.5
METRICS = ['sig_Homeostatic', 'sig_DAM', 'sig_DAMratio']
METHODS = ['c2l', 'stereo']

assert STEREO_CSV.exists(), f'pull stereoscope outputs from the peer first: {STEREO_CSV}'
c2l = pd.read_csv(C2L_CSV, index_col=0)
st  = pd.read_csv(STEREO_CSV, index_col=0)
noise = pd.read_csv(NOISE_CSV, index_col=0).iloc[:, 0] if NOISE_CSV.exists() else None
scores = pd.read_csv(SCORES_CSV, index_col=0)

spots = c2l.index.intersection(st.index).intersection(scores.index)
types = [t for t in c2l.columns if t in st.columns]
print(f'spots: c2l {len(c2l)}, stereo {len(st)}, scores {len(scores)} -> shared {len(spots)}')
print('shared cell types:', types)
c2l, st = c2l.loc[spots, types], st.loc[spots, types]

o = scores.loc[spots, [SAMPLE_KEY, TREAT_KEY, REG] + METRICS].copy()
o['slide'] = o[SAMPLE_KEY].astype(str).str.split('_').str[0]
o['mg_c2l'], o['mg_stereo'] = c2l[MG].clip(lower=0), st[MG].clip(lower=0)
for m in METHODS:
    x = o[f'mg_{m}']
    o[f'z_{m}'] = (x - x.mean()) / x.std()

## 1 · Per-spot agreement

In [ ]:
rows = []
for t in types:
    rows.append(dict(cell_type=t, mean_c2l=c2l[t].mean(), mean_stereo=st[t].mean(),
                     pearson=pearsonr(c2l[t], st[t])[0], spearman=spearmanr(c2l[t], st[t])[0]))
agree = pd.DataFrame(rows).set_index('cell_type').round(3)
agree.to_csv(OUT / 'c2l_vs_stereo_agreement.csv')
display(agree)
if noise is not None:
    print(f'stereoscope noise term: mean {noise.mean():.3f}, median {noise.median():.3f} (excluded from fractions)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
ax = axes[0]
hb = ax.hexbin(o['mg_c2l'], o['mg_stereo'], gridsize=60, bins='log', cmap='viridis', mincnt=1)
lim = max(o['mg_c2l'].quantile(.999), o['mg_stereo'].quantile(.999))
ax.plot([0, lim], [0, lim], ls='--', c='grey', lw=.8)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel('cell2location microglia fraction'); ax.set_ylabel('stereoscope microglia fraction')
ax.set_title(f"Per spot (r = {agree.loc[MG, 'pearson']:.2f}, rho = {agree.loc[MG, 'spearman']:.2f})", fontsize=10)
fig.colorbar(hb, ax=ax, label='log10 spots')

ax = axes[1]
x = np.arange(len(types)); w = .38
ax.bar(x - w/2, agree['mean_c2l'], w, label='cell2location', color='#4c72b0')
ax.bar(x + w/2, agree['mean_stereo'], w, label='stereoscope', color='#dd8452')
ax.set_xticks(x); ax.set_xticklabels(types, rotation=45, ha='right')
ax.set_ylabel('mean fraction / spot'); ax.set_title('Mean composition', fontsize=10); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(FIG / 'deconv_stereo_vs_c2l_agreement.png', dpi=200, bbox_inches='tight')
plt.show()

## 2 · Section-level microglia means

If the slide pattern is real tissue/batch biology and not a cell2location artifact, stereoscope should also put P24215 sections (WT, BRICHOS 201/301) low and BRICHOS 401 (on P28052) level with PBS.

In [ ]:
sec = (o.groupby([SAMPLE_KEY, TREAT_KEY, 'slide'], observed=True)[['mg_c2l', 'mg_stereo']]
         .mean().reset_index().sort_values([TREAT_KEY, SAMPLE_KEY]))
display(sec.round(4))
print('section-level Spearman c2l vs stereo:', round(spearmanr(sec['mg_c2l'], sec['mg_stereo'])[0], 3))

fig, ax = plt.subplots(figsize=(4.6, 4.2))
for (t, sl), d in sec.groupby([TREAT_KEY, 'slide']):
    ax.scatter(d['mg_c2l'], d['mg_stereo'], c=TCOL[t], marker='o' if sl == 'P28052' else '^',
               s=60, edgecolor='k', lw=.4, label=f'{t} / {sl}')
for _, r in sec.iterrows():
    ax.annotate(r[SAMPLE_KEY].split('_')[1], (r['mg_c2l'], r['mg_stereo']), fontsize=6,
                xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('c2l mean microglia fraction'); ax.set_ylabel('stereoscope mean microglia fraction')
ax.set_title('Per section (o = P28052, ^ = P24215)', fontsize=10)
ax.legend(fontsize=7, frameon=False)
fig.tight_layout(); fig.savefig(FIG / 'deconv_stereo_vs_c2l_sections.png', dpi=200, bbox_inches='tight')
plt.show()

for m in METHODS:
    x = sec.loc[sec[TREAT_KEY] == 'PBS', f'mg_{m}']; z = sec.loc[sec[TREAT_KEY] == 'BRICHOS', f'mg_{m}']
    print(f'{m:6s} whole-section microglia PBS {x.mean():.4f} vs BRICHOS {z.mean():.4f}  MWU p={mannwhitneyu(x, z).pvalue:.3f}')

## 3 · Hotspot overlap (z > 1.5)

In [ ]:
hot = {m: set(o.index[o[f'z_{m}'] > Z_THRESH]) for m in METHODS}
inter, union = hot['c2l'] & hot['stereo'], hot['c2l'] | hot['stereo']
print(f"hotspots: c2l {len(hot['c2l'])}, stereo {len(hot['stereo'])}, shared {len(inter)}, "
      f"Jaccard {len(inter) / len(union):.2f}")
display(pd.DataFrame({m: o.loc[list(hot[m]), TREAT_KEY].value_counts() for m in METHODS}).reindex(ORDER))

## 4 · Downstream state shift, recomputed with each method

(A) deconv-weighted section means (as in 10c-A): each spot's signature score is weighted by its microglia fraction; PBS vs BRICHOS Mann-Whitney on section means.
(B) hotspot mixed model `score ~ treatment + (1|section)`, and the slide-adjusted version `+ C(slide)` (as in 11). Scores are z-scored over all spots so betas are in SD units and comparable between methods.

In [ ]:
def wmean(df, col, wcol):
    w = df[wcol].values
    return np.nan if w.sum() == 0 else float(np.average(df[col].values, weights=w))

wrows = []
for m in METHODS:
    ws = pd.DataFrame([dict(sample_id=s, treatment=t, **{k: wmean(d, k, f'mg_{m}') for k in METRICS})
                       for (s, t), d in o.groupby([SAMPLE_KEY, TREAT_KEY], observed=True)])
    for k in METRICS:
        x = ws.loc[ws[TREAT_KEY] == 'PBS', k]; z = ws.loc[ws[TREAT_KEY] == 'BRICHOS', k]
        wrows.append(dict(method=m, metric=k, PBS=x.mean(), BRICHOS=z.mean(),
                          delta=z.mean() - x.mean(), MWU_p=mannwhitneyu(x, z).pvalue))
weighted = pd.DataFrame(wrows)
weighted.to_csv(OUT / 'c2l_vs_stereo_weighted.csv', index=False)
display(weighted.pivot(index='metric', columns='method', values=['delta', 'MWU_p']).round(4))

In [ ]:
zs = o.copy()
for k in METRICS:
    zs[k] = (zs[k] - zs[k].mean()) / zs[k].std()

def fit(df, k, slide):
    f = f"{k} ~ C({TREAT_KEY}, Treatment('PBS'))" + (' + C(slide)' if slide else '')
    r = smf.mixedlm(f, df, groups=df[SAMPLE_KEY]).fit(reml=True)
    term = [t for t in r.params.index if 'BRICHOS' in t][0]
    return r.params[term], r.pvalues[term]

mrows = []
for m in METHODS:
    h = zs[(zs[f'z_{m}'] > Z_THRESH) & zs[TREAT_KEY].isin(['PBS', 'BRICHOS'])]
    for k in METRICS:
        b0, p0 = fit(h, k, False); b1, p1 = fit(h, k, True)
        mrows.append(dict(method=m, metric=k, n_spots=len(h), beta_unadj=b0, p_unadj=p0,
                          beta_slideadj=b1, p_slideadj=p1))
mixed = pd.DataFrame(mrows)
mixed.to_csv(OUT / 'c2l_vs_stereo_hotspot_mixedmodel.csv', index=False)
display(mixed.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
labels = [k.replace('sig_', '') for k in METRICS]; y = np.arange(len(METRICS)); off = {'c2l': -.15, 'stereo': .15}
cols = {'c2l': '#4c72b0', 'stereo': '#dd8452'}
for ax, (b, p, title) in zip(axes, [('beta_unadj', 'p_unadj', 'hotspots, (1|section)'),
                                     ('beta_slideadj', 'p_slideadj', '+ slide fixed effect')]):
    for m in METHODS:
        d = mixed[mixed.method == m].set_index('metric').loc[METRICS]
        ax.scatter(d[b], y + off[m], c=cols[m], s=50, label=m, zorder=3)
        for yi, (bv, pv) in enumerate(zip(d[b], d[p])):
            ax.annotate(f'p={pv:.3f}', (bv, yi + off[m]), fontsize=7, xytext=(5, -3), textcoords='offset points')
    ax.axvline(0, c='grey', lw=.8, ls='--')
    ax.set_yticks(y); ax.set_yticklabels(labels); ax.set_title(title, fontsize=10)
    ax.set_xlabel('BRICHOS − PBS (SD units)')
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(FIG / 'deconv_stereo_vs_c2l_state_effect.png', dpi=200, bbox_inches='tight')
plt.show()

## Interpretation

*Fill in after running.* Things to note:
- microglia per-spot r and hotspot Jaccard (agreement on *where* microglia are),
- whether section ranking and the P24215-low pattern are reproduced (if yes, the slide effect isn't a c2l artifact),
- whether the direction of the homeostatic-up / DAM:homeostatic-down shift holds under stereoscope weights, and whether the slide-adjusted attenuation looks the same.

A second deconvolution method can't resolve the treatment–slide confound; see [batch-confound notes in 11](11-inflammation-results.ipynb).